# 🏦 Data-Driven Tabular Synthesis — Bank Marketing

In this notebook we will train a data-driven tabular synthesizer on a real dataset and generate synthetic records that preserve the original statistical patterns, then evaluate, constrain, and refine the output.

The dataset in questions is the [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), 45,211 records from a Portuguese bank's direct marketing campaigns. The target variable `y` indicates whether a client subscribed to a term deposit.

The notebook follows the structure below:
1. Load the dataset
2. Configure the synthesizer before training, including calculated features and anonymizer.
3. Train a `RegularSynthesizer` to learn the data distribution
4. Generate a synthetic dataset
5. Compare original vs synthetic to validate quality
6. Apply the `ConstraintEngine` to validate the presence of structural inconsistencies
7. Review the full pipeline

## 1. Load the dataset

### A note on the `duration` column

The dataset documentation flags `duration` (last call duration in seconds) as a **data leakage** variable: if `duration = 0`, the call never happened and `y` is always `'no'`. This variable is only known *after* a call ends, so it is not available at prediction time.

We drop it before training — a generative model that learns this near-deterministic relationship would embed the leakage into every synthetic record.

In [1]:
import pandas as pd

df = pd.read_csv("../data/bank-full.csv", sep=';') #read the dataset

In [2]:
df = df.drop(columns=['duration'])
print(f'Columns after dropping duration: {df.shape[1]}')
print(list(df.columns))

Columns after dropping duration: 16
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


## 2. Synthesizer - configuration & training

Before training the synthesizer,  2 important configurations can make the synthetic data more reliable. This is true for any generative model that you decide to use:

1. **Calculated features** — derive new columns from deterministic relationships in the data, so the synthesizer doesn't have to learn them probabilistically.
2. **Anonymization** — mask sensitive identifiers before the model ever sees them if you decide to keep them.

Both are passed directly to `RegularSynthesizer.fit()`, keeping the pipeline clean and reproducible.

### 2a. Calculated Features

The dataset has two columns that count prior contact history:
- `previous` — number of contacts made before this campaign
- `campaign` — number of contacts made during the current campaign

Their sum, `total_contacts`, is a deterministic relationship: it should always equal their sum `campaign + previous`. 
Rather than hoping the model to learn this arithmetic, we define it explicitly as a calculated feature — the synthesizer will enforce it rather than approximate it.

In [3]:
# Define the calculated feature function
def get_total_contacts(campaign, previous):
    return (campaign + previous).values

# Package it in the format expected by RegularSynthesizer
calculated_features = [
    {
        'calculated_features': 'total_contacts',   # name of the new column
        'function': get_total_contacts,
        'calculated_from': ['campaign', 'previous'],
    }
]

print('Calculated features defined ✓')
print(f'  total_contacts = campaign + previous')
print(f'  Example: {df["campaign"].iloc[0]} + {df["previous"].iloc[0]} = {df["campaign"].iloc[0] + df["previous"].iloc[0]}')

Calculated features defined ✓
  total_contacts = campaign + previous
  Example: 1 + 0 = 1


### 2b. Anonymization

The public Bank Marketing dataset has no direct identifiers — names, phone numbers, and account IDs were stripped before release. In a real banking dataset, however, the raw data would contain them.

The `anonymize` parameter in `RegularSynthesizer.fit()` lets you specify which columns to replace before training, so those values are never seen by the model. Below is the pattern; adapt the column list to your actual dataset.

In [4]:
# In a real bank dataset you would anonymize columns like:
#
#   anonymize = {
#       'client_name':  {'faker': 'name'},
#       'phone_number': {'faker': 'phone_number'},
#       'client_id':    {'faker': 'uuid4'},
#   }
#
# These columns are replaced BEFORE the synthesizer trains,
# so the model never memorises real identifiers.
#
# For this public dataset we pass None — no anonymization needed.

anonymize = None

print('Anonymization configured ✓')
print('  (no PII columns in the public dataset — pattern shown above for reference)')

Anonymization configured ✓
  (no PII columns in the public dataset — pattern shown above for reference)


### 2c. Train the RegularSynthesizer

`RegularSynthesizer` uses deep generative models to learn the **joint distribution** of the dataset: not just individual column ranges, but how columns relate to each other.

First, lets print the metadata.

In [5]:
from ydata.dataset import Dataset
from ydata.metadata import Metadata
from ydata.synthesizers.regular import RegularSynthesizer

# Wrap the dataframe
dataset  = Dataset(df)
metadata = Metadata(dataset)

print('Metadata summary:')
print(metadata)

INFO: 2026-08-12 13:31:55,569 Using device of type: cpu


Done:  80%|████████████████████████████▊       | 4/5 [00:00<00:00,  4.87step/s]

Metadata summary:
Metadata Summary 
 
Dataset type: TABULAR
Dataset attributes: 
Number of columns: 16
Number of rows: 45211
Duplicate rows: 16
Target column: 

Column detail: 
       Column    Data type Variable type Characteristics
0         age    numerical           int                
1         job  categorical        string                
2     marital  categorical        string                
3   education  categorical        string                
4     default  categorical        string                
5     balance    numerical           int                
6     housing  categorical        string                
7        loan  categorical        string                
8     contact  categorical        string                
9         day    numerical           int                
10      month  categorical        string                
11   campaign    numerical           int                
12      pdays    numerical           int                
13   previous    numerica

Calling `fit()` starts the training process. The model learns numerical ranges, category frequencies, and cross-feature interactions in a single pass.

In [6]:
synth = RegularSynthesizer()

synth.fit(
    X=dataset,
    metadata=metadata,
    calculated_features=calculated_features,
)

print('Training complete ✓')

Validate the segmentation strategy.
INFO: 2026-08-12 13:31:58,516 [SYNTHESIZER] - Number columns considered for synth: 16
INFO: 2026-08-12 13:31:58,968 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-12 13:31:58,970 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-12 13:31:58,973 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-12 13:31:58,974 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
Training complete ✓


## 3. Generate Synthetic Data

Once trained, `sample()` generates as many rows as needed. The model draws from the learned distribution — each synthetic record is statistically consistent with the original data but has no 1-to-1 relationship with any real individual.

In [7]:
sample = synth.sample(n_samples=len(df))
synth_df = sample.to_pandas()

print(f'Generated: {synth_df.shape[0]:,} rows × {synth_df.shape[1]} columns ✓')
synth_df.head()

INFO: 2026-08-12 13:32:05,119 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-12 13:32:06,479 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-12 13:32:06,483 [SYNTHESIZER] - Numerical clipping
INFO: 2026-08-12 13:32:06,514 let write into the connector None
INFO: 2026-08-12 13:32:06,514 using kwargs {}
Generated: 45,211 rows × 17 columns ✓


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y,total_contacts
0,34,unemployed,single,tertiary,no,1974,no,no,cellular,14,aug,2,-1,0,unknown,no,2
1,59,retired,married,secondary,no,11,no,no,cellular,18,jul,10,-1,0,unknown,no,10
2,61,retired,married,tertiary,no,448,yes,no,cellular,1,apr,1,-1,0,unknown,no,1
3,46,blue-collar,single,secondary,no,2401,yes,no,unknown,14,may,2,-1,0,unknown,no,2
4,31,management,single,tertiary,no,730,yes,no,cellular,27,may,2,89,1,failure,no,3


## 4. Compare Original vs Synthetic

> **⏸ Active Learning** — Before running the cells below, look at the synthetic sample above. Do the values look plausible? Do the categories match what you'd expect? Then run the comparison and check: where does the synthetic data track the original closely, and where does it diverge?

In [8]:
from ydata.profiling import ProfileReport

original_report = ProfileReport(df, title='Real dataset profiling')
synthetic_report = ProfileReport(synth_df, title='Synthetic data profiling')

compare = original_report.compare(synthetic_report)
compare


INFO: 2026-08-12 13:32:08,719 Pandas backend loaded 2.3.3
INFO: 2026-08-12 13:32:08,721 Numpy backend loaded 2.3.5
INFO: 2026-08-12 13:32:08,721 Pyspark backend NOT loaded
INFO: 2026-08-12 13:32:08,721 Python backend loaded


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|████████████████████████████████████████| 16/16 [00:00<00:00, 2396.66it/s]


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|████████████████████████████████████████| 16/16 [00:00<00:00, 2383.55it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
synth_df.to_csv('data/bank-full_synthetic.csv', index=False)

## 5. Structural Hallucinations & the Constraint Engine

Generative models are probabilistic, they learn patterns, but they don't interpret business logic. This can occasionally produce **structural hallucinations**: records that are statistically plausible but logically inconsistent.

For this dataset, three rules should always hold:

| Rule | Logic |
|---|---|
| No prior contact → no prior outcome | If `pdays == -1` (never contacted), `poutcome` must be `'unknown'` |
| Contact count must be positive | `campaign >= 1` — at least one contact was made |
| Age must be adult | `age >= 18` |

Let's test them out! 

In [15]:
from ydata.constraints import (
    ConstraintEngine,
    GreaterThan,
    CustomConstraint,
)
from ydata.dataset import Dataset

engine = ConstraintEngine([
    # At least one campaign contact must have been made
    GreaterThan(
        columns=['campaign'],
        value=0,
    ),

    # Clients must be adults
    GreaterThan(
        columns=['age'],
        value=17,
    ),

    # If never previously contacted, outcome must be unknown
    CustomConstraint(
        lambda row: not (row['pdays'] == -1 and row['poutcome'] != 'unknown'),
        columns=['pdays', 'poutcome'],
        name='no_contact_no_outcome',
        axis='row',
    ),
])

engine.explain_constraints()
print('Constraint engine ready ✓')

Constraint engine ready ✓


In [16]:
synth_dataset = Dataset(synth_df)

engine.validate(synth_dataset)
print('=== Constraint validation summary ===')
engine.summary()

=== Constraint validation summary ===


{'rows_violation_count': np.int64(0),
 'rows_violation_ratio': 0.0,
 'violation_per_constraint': {"GreaterThan(columns=['campaign'], value=0)": {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0},
  "GreaterThan(columns=['age'], value=17)": {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0},
  'no_contact_no_outcome': {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0}}}

In [17]:
clean_synth_df = engine.filter(synth_dataset).to_pandas()

print(f'Rows before constraint filtering: {len(synth_df):,}')
print(f'Rows after constraint filtering:  {len(clean_synth_df):,}')
clean_synth_df.head()

Rows before constraint filtering: 45,211
Rows after constraint filtering:  45,211


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y,total_contacts
0,36,blue-collar,married,primary,no,1503,yes,no,cellular,11,may,1,-1,0,unknown,yes,1
1,47,technician,married,secondary,no,0,yes,no,unknown,28,may,3,-1,0,unknown,no,3
2,34,blue-collar,married,secondary,no,1221,yes,no,cellular,17,apr,2,331,2,failure,no,4
3,45,admin.,married,secondary,no,0,no,no,cellular,22,aug,6,-1,0,unknown,no,6
4,43,blue-collar,divorced,secondary,no,-259,no,yes,cellular,14,jul,2,-1,0,unknown,no,2


## 6. The Full Pipeline

The steps above form a repeatable pipeline for responsible synthetic data generation:

```
Raw data
    │
    ├─► Drop leakage columns (duration)
    │
    ├─► Anonymize sensitive identifiers    ← before the model sees the data
    │
    ├─► Define calculated features         ← enforce deterministic relationships
    │
    ├─► RegularSynthesizer.fit()           ← learn the joint distribution
    │
    ├─► .sample(n_samples)                 ← generate synthetic records
    │
    └─► ConstraintEngine.filter()          ← remove any structural inconsistencies
            │
            └─► Final synthetic dataset ✓
```

Each step addresses a different failure mode: leakage, privacy, arithmetic consistency, distributional quality, and logical validity.

In [18]:
output_path = 'bank_marketing_synthetic.csv'
clean_synth_df.to_csv(output_path, index=False)
print(f'Dataset saved → {output_path} ({len(clean_synth_df):,} rows) ✓')

Dataset saved → bank_marketing_synthetic.csv (45,211 rows) ✓


## ✅ Summary

In this notebook you:

1. **Loaded** the Bank Marketing dataset and removed the `duration` leakage column before training
2. **Defined a calculated feature** (`total_contacts = campaign + previous`) so the synthesizer enforces arithmetic consistency rather than approximating it
3. **Configured anonymization** — the pattern for masking sensitive identifiers before the model trains
4. **Trained a `RegularSynthesizer`** to learn the joint distribution of the dataset
5. **Generated 45,000+ synthetic records** and compared them to the original
6. **Validate and filter** to  out any structurally inconsistent records